In [1]:
# 🔍 Deepfake Detection System
### Final Year Project — Kaggle Notebook

**Architecture:** Dual-Branch (EfficientNet-B4 RGB + FFT CNN)  
**Dataset:** 140k Real and Fake Faces (already available in Kaggle)  
**Features:**
- ✅ No Google Drive needed — everything stays in Kaggle
- ✅ Checkpoint saving to `/kaggle/working/` after every epoch
- ✅ Full training history saved to JSON
- ✅ Resume from checkpoint if session ends
- ✅ Mixed precision training (faster)
- ✅ Grad-CAM explainability
- ✅ Full evaluation metrics

---
**Before running:**
1. Add dataset: Click **+ Add Data** → search `140k real and fake faces` → add it
2. Enable GPU: Settings → Accelerator → **GPU T4 x2**
3. Run all cells top to bottom


SyntaxError: invalid decimal literal (3501118350.py, line 5)

## ✅ Cell 1 — Install Libraries

In [ ]:
!pip install -q timm grad-cam albumentations
print('✅ Libraries installed')

In [ ]:
import os

# Find exact dataset path
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth < 3:
        indent = '  ' * depth
        print(f'{indent}{root}')

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input/datasets/xhlulu'):
    depth = root.replace('/kaggle/input/datasets/xhlulu', '').count(os.sep)
    if depth < 4:
        indent = '  ' * depth
        print(f'{indent}{root}')

## ✅ Cell 2 — Setup Paths & Folders

In [ ]:
import os

# ── Kaggle paths ─────────────────────────────────────────────────
WORKING_DIR    = '/kaggle/working'
MODEL_DIR      = os.path.join(WORKING_DIR, 'models')
HISTORY_DIR    = os.path.join(WORKING_DIR, 'history')
RESULTS_DIR    = os.path.join(WORKING_DIR, 'results')
CHECKPOINT_DIR = os.path.join(WORKING_DIR, 'checkpoints')

for d in [MODEL_DIR, HISTORY_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Fixed dataset path ───────────────────────────────────────────
DATA_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'

if os.path.exists(DATA_DIR):
    print(f'✅ Dataset found at: {DATA_DIR}')
    print('\nFolder structure:')
    for item in os.listdir(DATA_DIR):
        folder_path = os.path.join(DATA_DIR, item)
        if os.path.isdir(folder_path):
            subfolders = os.listdir(folder_path)
            count = sum(len(files) for _, _, files in os.walk(folder_path))
            print(f'  {item}/ → {count:,} images')
            for sub in subfolders:
                sub_path = os.path.join(folder_path, sub)
                if os.path.isdir(sub_path):
                    n = len(os.listdir(sub_path))
                    print(f'    {sub}/ → {n:,} images')
else:
    print(f'❌ Dataset not found at: {DATA_DIR}')
    print('Available:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 5:
            print(f'  {"  "*depth}{root}')

## ✅ Cell 3 — Config

In [ ]:
import torch

# ── Check if dataset has valid/ or only train/ and test/ ─────────
has_valid = os.path.exists(os.path.join(DATA_DIR, 'valid'))
val_folder = 'valid' if has_valid else 'test'
print(f'Using "{val_folder}" folder for validation')

# ════════════════════════════════════════════════════════════════
CFG = {
    # Paths
    'train_dir'  : os.path.join(DATA_DIR, 'train'),
    'val_dir'    : os.path.join(DATA_DIR, val_folder),
    'test_dir'   : os.path.join(DATA_DIR, 'test'),

    # Model
    'model_name' : 'efficientnet_b4',
    'img_size'   : 224,
    'num_classes': 2,

    # Training
    'epochs'         : 12,
    'batch_size'     : 32,
    'lr'             : 2e-4,
    'weight_decay'   : 1e-4,
    'unfreeze_epoch' : 5,
    'label_smoothing': 0.1,
    'mixup_alpha'    : 0.2,

    # Scheduler
    'warmup_epochs': 2,

    # Early stopping
    'patience': 5,

    # Misc
    'num_workers': 0,
    'seed'       : 42,
    'device'     : 'cuda' if torch.cuda.is_available() else 'cpu',
}

# File paths
HISTORY_FILE    = os.path.join(HISTORY_DIR,    'training_history.json')
BEST_MODEL_PATH = os.path.join(MODEL_DIR,      'best_model.pth')
LAST_CKPT_PATH  = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pth')

print(f"Device  : {CFG['device']}")
if CFG['device'] == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print('⚠️  No GPU! Go to Settings → Accelerator → GPU T4 x2')

## ✅ Cell 4 — Imports & Reproducibility

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import timm
import numpy as np
import json
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, f1_score, accuracy_score
)
from tqdm.auto import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG['seed'])
DEVICE = torch.device(CFG['device'])
print('✅ All imports done, seed set to', CFG['seed'])

## ✅ Cell 5 — Augmentation & Dataset

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_aug = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.GaussianBlur(blur_limit=(3,7), p=0.2),
    A.GaussNoise(p=0.2),
    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

val_aug = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset      = datasets.ImageFolder(root_dir)
        self.transform    = transform
        self.classes      = self.dataset.classes
        self.class_to_idx = self.dataset.class_to_idx

    def __len__(self):
        return len(self.dataset)

    def compute_fft_features(self, img_tensor):
        gray      = img_tensor.mean(dim=0).numpy()
        fft       = np.fft.fft2(gray)
        fft_shift = np.fft.fftshift(fft)
        magnitude = np.log(np.abs(fft_shift) + 1e-8)
        magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
        return torch.tensor(magnitude, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)

        if self.transform:
            aug    = self.transform(image=img)
            tensor = aug['image']
        else:
            tensor = torch.tensor(img).permute(2,0,1).float() / 255.0

        fft_tensor = self.compute_fft_features(tensor)
        return tensor, fft_tensor, label


train_ds = DeepfakeDataset(CFG['train_dir'], transform=train_aug)
val_ds   = DeepfakeDataset(CFG['val_dir'],   transform=val_aug)
test_ds  = DeepfakeDataset(CFG['test_dir'],  transform=val_aug)

# Weighted sampler for class balance
labels  = [s[1] for s in train_ds.dataset.samples]
counts  = np.bincount(labels)
weights = 1.0 / counts[labels]
sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          sampler=sampler, num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

print(f'✅ Datasets loaded:')
print(f'   Train : {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'   Classes: {train_ds.class_to_idx}')
print(f'   Fake: {counts[0]:,} | Real: {counts[1]:,}')

## ✅ Cell 6 — Model Architecture

In [ ]:
class DualBranchDeepfakeDetector(nn.Module):
    """
    Branch 1 (RGB)  — EfficientNet-B4: spatial & texture features
    Branch 2 (FFT)  — Lightweight CNN: frequency domain features
    Fused → classifier head
    """
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        # Branch 1: EfficientNet-B4
        self.rgb_backbone = timm.create_model(
            'efficientnet_b4', pretrained=pretrained, num_classes=0
        )
        rgb_features = self.rgb_backbone.num_features  # 1792

        # Branch 2: Lightweight FFT CNN
        self.fft_branch = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def freeze_backbone(self):
        for param in self.rgb_backbone.parameters():
            param.requires_grad = False
        print('  Backbone frozen')

    def unfreeze_backbone(self):
        for param in self.rgb_backbone.parameters():
            param.requires_grad = True
        print('  ✓ All layers unfrozen')

    def forward(self, rgb, fft):
        rgb_feat = self.rgb_backbone(rgb)
        fft_feat = self.fft_branch(fft).flatten(1)
        return self.classifier(torch.cat([rgb_feat, fft_feat], dim=1))


model = DualBranchDeepfakeDetector(num_classes=CFG['num_classes']).to(DEVICE)
model.freeze_backbone()

total_p    = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model ready — Total: {total_p/1e6:.1f}M | Trainable: {trainable_p/1e6:.1f}M')

## ✅ Cell 7 — Optimizer & Scheduler

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

# Check if resuming — adjust LR accordingly
resume_epoch = 0
if os.path.exists(LAST_CKPT_PATH):
    ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)
    resume_epoch = ckpt['epoch'] + 1
    print(f'Checkpoint found at epoch {ckpt["epoch"]} — adjusting LR')

if resume_epoch >= CFG['unfreeze_epoch']:
    model.unfreeze_backbone()
    effective_lr = CFG['lr'] * 0.1
else:
    effective_lr = CFG['lr']

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=effective_lr,
    weight_decay=CFG['weight_decay']
)

remaining_epochs = max(CFG['epochs'] - resume_epoch, 1)
scheduler = OneCycleLR(
    optimizer,
    max_lr=effective_lr,
    steps_per_epoch=len(train_loader),
    epochs=remaining_epochs,
    pct_start=0.1,
    final_div_factor=100,
)

print(f'✅ Optimizer ready | LR: {effective_lr} | Remaining epochs: {remaining_epochs}')

## ✅ Cell 8 — Checkpoint & History Utilities

In [ ]:
def save_checkpoint(epoch, model, optimizer, scheduler, best_auc, history):
    torch.save({
        'epoch'          : epoch,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_auc'       : best_auc,
    }, LAST_CKPT_PATH)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

def load_checkpoint():
    if os.path.exists(LAST_CKPT_PATH):
        ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auc    = ckpt['best_auc']
        print(f'✅ Resumed from epoch {ckpt["epoch"]}, best AUC={best_auc:.4f}')
        return start_epoch, best_auc
    print('No checkpoint found — starting fresh')
    return 0, 0.0

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE) as f:
            h = json.load(f)
        print(f'✅ History loaded: {len(h["epoch"])} epochs recorded')
        return h
    return {'epoch':[], 'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_auc':[], 'val_f1':[], 'lr':[]}

def mixup_data(x_rgb, x_fft, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x_rgb.size(0)).to(DEVICE)
    return (lam*x_rgb + (1-lam)*x_rgb[idx],
            lam*x_fft + (1-lam)*x_fft[idx],
            y, y[idx], lam)

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)

print('✅ Utilities ready')
print(f'   Checkpoint : {LAST_CKPT_PATH}')
print(f'   History    : {HISTORY_FILE}')
print(f'   Best model : {BEST_MODEL_PATH}')

## 🚀 Cell 9 — MAIN TRAINING LOOP

In [ ]:
start_epoch, best_auc = load_checkpoint()
history          = load_history()
patience_counter = 0
scaler           = torch.cuda.amp.GradScaler()

print(f'\n🚀 Training from epoch {start_epoch+1}/{CFG["epochs"]}')
print('─' * 70)

for epoch in range(start_epoch, CFG['epochs']):
    epoch_start = time.time()

    # Unfreeze backbone at configured epoch
    if epoch == CFG['unfreeze_epoch']:
        model.unfreeze_backbone()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CFG['lr']*0.1, weight_decay=CFG['weight_decay'])
        scheduler = OneCycleLR(
            optimizer, max_lr=CFG['lr']*0.1,
            steps_per_epoch=len(train_loader),
            epochs=CFG['epochs']-epoch,
            pct_start=0.1, final_div_factor=100)

    # ── Train ────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Ep {epoch+1:02d}/{CFG["epochs"]} [Train]', leave=False)

    for rgb, fft, labels in pbar:
        rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
        rgb_m, fft_m, y_a, y_b, lam = mixup_data(rgb, fft, labels, CFG['mixup_alpha'])

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out  = model(rgb_m, fft_m)
            loss = mixup_criterion(criterion, out, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for rgb, fft, labels in tqdm(val_loader, desc=f'Ep {epoch+1:02d}/{CFG["epochs"]} [Val]  ', leave=False):
            rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                out  = model(rgb, fft)
                loss = criterion(out, labels)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)[:,1].cpu().numpy()
            preds = out.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    val_acc   = accuracy_score(all_labels, all_preds)
    val_auc   = roc_auc_score(all_labels, all_probs)
    val_f1    = f1_score(all_labels, all_preds)
    cur_lr    = optimizer.param_groups[0]['lr']
    elapsed   = time.time() - epoch_start

    # Log history
    history['epoch'].append(epoch+1)
    history['train_loss'].append(round(avg_train, 5))
    history['val_loss'].append(round(avg_val, 5))
    history['val_acc'].append(round(val_acc, 5))
    history['val_auc'].append(round(val_auc, 5))
    history['val_f1'].append(round(val_f1, 5))
    history['lr'].append(cur_lr)

    star = '⭐' if val_auc > best_auc else '  '
    print(f'Ep {epoch+1:02d}/{CFG["epochs"]} {star} | '
          f'Loss {avg_train:.4f}→{avg_val:.4f} | '
          f'Acc {val_acc:.4f} | AUC {val_auc:.4f} | '
          f'F1 {val_f1:.4f} | LR {cur_lr:.2e} | {elapsed:.0f}s')

    # Save best model
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'model_state': model.state_dict(),
            'val_auc'    : val_auc,
            'val_acc'    : val_acc,
            'epoch'      : epoch+1,
            'cfg'        : CFG,
        }, BEST_MODEL_PATH)
        patience_counter = 0
        print(f'   ✅ Best model saved (AUC={val_auc:.4f})')
    else:
        patience_counter += 1

    # Save checkpoint every epoch
    save_checkpoint(epoch, model, optimizer, scheduler, best_auc, history)

    # Early stopping
    if patience_counter >= CFG['patience']:
        print(f'\n⚡ Early stopping at epoch {epoch+1}')
        break

print(f'\n✅ Training complete! Best AUC = {best_auc:.4f}')

## ✅ Cell 10 — Training Curves

In [ ]:
with open(HISTORY_FILE) as f:
    h = json.load(f)

epochs = h['epoch']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

axes[0].plot(epochs, h['train_loss'], label='Train', color='#534AB7')
axes[0].plot(epochs, h['val_loss'],   label='Val',   color='#D85A30')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, h['val_acc'], label='Accuracy', color='#1D9E75')
axes[1].plot(epochs, h['val_auc'], label='AUC',      color='#E8A838')
axes[1].plot(epochs, h['val_f1'],  label='F1',       color='#534AB7')
axes[1].set_title('Metrics'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, h['lr'], color='#534AB7')
axes[2].set_title('Learning Rate')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to results/training_curves.png')

## ✅ Cell 11 — Full Evaluation on Test Set

In [ ]:
ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'✅ Loaded best model — epoch {ckpt["epoch"]}, AUC={ckpt["val_auc"]:.4f}')

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for rgb, fft, labels in tqdm(test_loader, desc='Testing'):
        rgb, fft = rgb.to(DEVICE), fft.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = model(rgb, fft)
        probs = torch.softmax(out, dim=1)[:,1].cpu().numpy()
        preds = out.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_auc = roc_auc_score(all_labels, all_probs)
test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds)

print('\n' + '='*50)
print('        FINAL TEST SET RESULTS')
print('='*50)
print(f'  Accuracy : {test_acc*100:.2f}%')
print(f'  AUC-ROC  : {test_auc:.4f}')
print(f'  F1 Score : {test_f1:.4f}')
print('='*50)
print()
print(classification_report(all_labels, all_preds, target_names=['Fake','Real']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake','Real'], yticklabels=['Fake','Real'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color='#534AB7', lw=2, label=f'AUC={test_auc:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#534AB7')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to results/evaluation.png')

## ✅ Cell 12 — Grad-CAM Heatmaps

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def run_gradcam(img_tensor, fft_tensor, label, class_names):
    rgb_in = img_tensor.unsqueeze(0).to(DEVICE)
    fft_in = fft_tensor.unsqueeze(0).to(DEVICE)

    class ModelWrapper(nn.Module):
        def __init__(self, m, f): super().__init__(); self.m=m; self.f=f
        def forward(self, x): return self.m(x, self.f)

    wrapper = ModelWrapper(model, fft_in)
    cam     = GradCAM(model=wrapper, target_layers=[wrapper.m.rgb_backbone.blocks[-1][-1]])

    with torch.no_grad():
        out   = model(rgb_in, fft_in)
        probs = torch.softmax(out, dim=1)[0]
        pred  = out.argmax(dim=1).item()

    grayscale = cam(input_tensor=rgb_in, targets=[ClassifierOutputTarget(0)])[0]

    mean    = np.array([0.485,0.456,0.406])
    std     = np.array([0.229,0.224,0.225])
    img_vis = np.clip(img_tensor.permute(1,2,0).numpy()*std+mean, 0, 1).astype(np.float32)
    heatmap = show_cam_on_image(img_vis, grayscale, use_rgb=True)

    return {
        'original': img_vis, 'heatmap': heatmap,
        'pred': class_names[pred], 'true': class_names[label],
        'fake_pct': probs[0].item()*100, 'real_pct': probs[1].item()*100,
        'correct': pred==label
    }

class_names = ['Fake', 'Real']
samples = {'Fake': [], 'Real': []}
for rgb, fft, lbl in test_ds:
    name = class_names[lbl]
    if len(samples[name]) < 4:
        samples[name].append((rgb, fft, lbl))
    if all(len(v)==4 for v in samples.values()):
        break

all_samples = samples['Fake'] + samples['Real']
results     = [run_gradcam(r, f, l, class_names) for r,f,l in all_samples]

fig, axes = plt.subplots(8, 2, figsize=(10, 32))
fig.suptitle('Grad-CAM Heatmaps — Suspicious Regions', fontsize=13, fontweight='bold', y=1.01)

for i, r in enumerate(results):
    color = '#1D9E75' if r['correct'] else '#D85A30'
    mark  = '✓' if r['correct'] else '✗'
    axes[i][0].imshow(r['original']); axes[i][0].set_title(f'True: {r["true"]}', fontsize=10); axes[i][0].axis('off')
    axes[i][1].imshow(r['heatmap'])
    axes[i][1].set_title(f'{mark} Pred:{r["pred"]} | Fake:{r["fake_pct"]:.1f}% Real:{r["real_pct"]:.1f}%',
                         fontsize=10, color=color)
    axes[i][1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gradcam.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to results/gradcam.png')

## ✅ Cell 13 — Export Final Model

In [ ]:
export_path = os.path.join(MODEL_DIR, 'deepfake_detector_final.pth')
ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)

torch.save({
    'model_state'  : ckpt['model_state'],
    'architecture' : 'DualBranchDeepfakeDetector',
    'backbone'     : 'efficientnet_b4',
    'num_classes'  : 2,
    'img_size'     : 224,
    'val_auc'      : ckpt['val_auc'],
    'val_acc'      : ckpt['val_acc'],
    'trained_epoch': ckpt['epoch'],
    'class_names'  : ['fake', 'real'],
}, export_path)

size_mb = os.path.getsize(export_path)/1e6
print(f'✅ Final model exported!')
print(f'   Path       : {export_path}')
print(f'   Size       : {size_mb:.1f} MB')
print(f'   Val AUC    : {ckpt["val_auc"]:.4f}')
print(f'   Val Acc    : {ckpt["val_acc"]*100:.2f}%')
print(f'   Epochs     : {ckpt["epoch"]}')
print()
print('Download this file from /kaggle/working/models/ for your web app')

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/models/deepfake_detector_final.pth')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/project_outputs', 'zip', '/kaggle/working')
print('✅ Zipped! Now download project_outputs.zip')

from IPython.display import FileLink
FileLink('/kaggle/working/project_outputs.zip')

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        full_path = os.path.join(root, f)
        size = os.path.getsize(full_path) / 1e6
        print(f'{full_path}  ({size:.1f} MB)')

In [ ]:
from IPython.display import FileLink
FileLink('project_outputs.zip')

In [ ]:
import shutil
from IPython.display import HTML
import base64

# For files under ~100MB, encode as base64 download link
file_path = 'models/deepfake_detector_final.pth'
with open(file_path, 'rb') as f:
    data = f.read()
b64 = base64.b64encode(data).decode()

html = f'<a download="deepfake_detector_final.pth" href="data:application/octet-stream;base64,{b64}">⬇ Click here to download model</a>'
HTML(html)